# 01 · BRONZE — Ingestão Boi Gordo (CEPEA CSV)

**Fonte:** CEPEA — Centro de Estudos Avançados em Economia Aplicada (USP)  
**URL download:** https://www.cepea.org.br/br/indicador/boi-gordo.aspx  
**Formato:** Excel (.xlsx) com colunas: Data (MM/yyyy), Valor (R$/arroba), Variação (%)  
**Destino:** `etl_pos__bronze.boi_gordo` (Delta Lake · overwrite)  
**SPEC:** SPEC_BRONZE.md — schema: Data (String), Valor (String), data_coleta (Timestamp)

---
### Pré-requisito — Upload do CSV/Excel CEPEA

**ADR-001:** A CEPEA não oferece API pública. Faça o download manual:

1. Acesse: https://www.cepea.org.br/br/indicador/boi-gordo.aspx
2. Filtre: 2024-01 → hoje
3. Exporte como Excel ou CSV
4. Renomeie para `boi_gordo.csv`
5. Upload via Databricks: **Data > DBFS > FileStore > etl_pos > bronze**
   - Path final: `dbfs:/FileStore/etl_pos/bronze/boi_gordo.csv`

> **PRÉ-REQUISITO:** Executar `00.config/config.ipynb` antes deste notebook.

In [ ]:
# ============================================================
# CELL 1 — Imports e constantes
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

SOURCE_PATH  = "dbfs:/FileStore/etl_pos/bronze/boi_gordo.csv"
TARGET_TABLE = "etl_pos__bronze.boi_gordo"

print(f"Fonte  : {SOURCE_PATH}")
print(f"Destino: {TARGET_TABLE}")

In [ ]:
# ============================================================
# CELL 2 — Verificar existência do arquivo no DBFS
# ============================================================
try:
    files = dbutils.fs.ls("dbfs:/FileStore/etl_pos/bronze/")
    print("Arquivos encontrados:")
    for f in files:
        print(f"  {f.path}  ({f.size} bytes)")
    
    boi_exists = any("boi_gordo" in f.path for f in files)
    if not boi_exists:
        raise FileNotFoundError(
            "boi_gordo.csv NAO encontrado em dbfs:/FileStore/etl_pos/bronze/\n"
            "Siga as instrucoes de upload no cabecalho deste notebook."
        )
    print("\nboi_gordo.csv encontrado — OK")
except Exception as e:
    print(f"ERRO: {e}")
    raise

In [ ]:
# ============================================================
# CELL 3 — Leitura do CSV CEPEA
# NOTA: inferSchema=True permite leitura flexível do formato
#       CEPEA. Normalização de tipos ocorre na camada Silver.
# ============================================================
df_spark = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "UTF-8")
    .option("sep", ";")
    .csv(SOURCE_PATH)
)

print("Schema inferido do CSV:")
df_spark.printSchema()
print(f"Total de linhas: {df_spark.count()}")
print(f"Colunas: {df_spark.columns}")

In [ ]:
# ============================================================
# CELL 4 — Normalização mínima Bronze + timestamp ingestão
# Bronze = dados brutos. Apenas rename de colunas se necessário
# e adição de data_coleta. Transformações ficam no Silver.
# ============================================================

# Renomear para nomes padronizados (caso o CSV CEPEA use variação)
# Ajustar conforme as colunas reais do seu download
col_map = {
    "Data"  : "data_cepea",
    "Valor" : "valor_cepea",
    # Adicione aqui outras colunas se o CEPEA incluir
}

for old_name, new_name in col_map.items():
    if old_name in df_spark.columns:
        df_spark = df_spark.withColumnRenamed(old_name, new_name)

# Adicionar metadata de ingestão
df_spark = df_spark.withColumn("data_coleta", F.current_timestamp())
df_spark = df_spark.withColumn("fonte", F.lit("CEPEA_MANUAL_UPLOAD"))

print("Schema final Bronze:")
df_spark.printSchema()

In [ ]:
# ============================================================
# CELL 5 — Preview
# ============================================================
display(df_spark)

In [ ]:
# ============================================================
# CELL 6 — Gravar Bronze (Delta · overwrite)
# ADR-003: sem path explícito no Databricks CE.
# ============================================================
(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela '{TARGET_TABLE}' gravada com sucesso!")

In [ ]:
# ============================================================
# CELL 7 — Validação (critérios TASK-002)
# ============================================================
df_val = spark.table(TARGET_TABLE)
total  = df_val.count()

print("=" * 50)
print("VALIDACAO — etl_pos__bronze.boi_gordo")
print("=" * 50)
print(f"  Total de registros: {total}")
print(f"  Colunas presentes : {df_val.columns}")
print(f"  Criterio >= 12    : {'OK' if total >= 12 else 'FALHOU'}")
print(f"  Colunas ok        : {'OK' if 'data_cepea' in df_val.columns else 'VERIFICAR NOMES'}")

display(df_val.limit(10))